In [ ]:
# save as generate_local_shap.py and run in Colab or local machine
import os, pickle, importlib.util, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")

DATA_PATH = "92da2bec-b99c-4f8e-91ad-b02d9438d5be.csv"  # adjust path if needed
OUT_DIR = "local_shap_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# Load data
df = pd.read_csv(DATA_PATH)
target = "defaulted" if "defaulted" in df.columns else df.columns[-1]
X = df.drop(columns=[target])
y = df[target].astype(int)

# Impute numeric missing values
imp = SimpleImputer(strategy="median")
X_imp = pd.DataFrame(imp.fit_transform(X), columns=X.columns)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X_imp, y, test_size=0.25, random_state=42, stratify=y)

# Try to load saved model files if you trained before (optional)
candidate_model_paths = [
    "final_model_XGBoost.pkl",
    "final_model_GradientBoosting.pkl",
    "model_outputs/final_model_XGBoost.pkl",
    "model_outputs/final_model_GradientBoosting.pkl",
    "final_model_LogisticRegression.pkl",
    "model_outputs/final_model_LogisticRegression.pkl"
]
model = None; scaler = None; model_name = None
for p in candidate_model_paths:
    if os.path.exists(p):
        try:
            with open(p, "rb") as f:
                model = pickle.load(f)
            scaler_path = p.replace("final_model_", "scaler_")
            if os.path.exists(scaler_path):
                with open(scaler_path, "rb") as f:
                    scaler = pickle.load(f)
            model_name = os.path.basename(p).replace("final_model_","").replace(".pkl","")
            break
        except Exception:
            model = None

# If no saved model found, train a quick tree model (GradientBoosting) to explain
if model is None:
    from sklearn.ensemble import GradientBoostingClassifier
    scaler = StandardScaler().fit(X_train)
    model = GradientBoostingClassifier(n_estimators=200, max_depth=4, random_state=42)
    model.fit(scaler.transform(X_train), y_train)
    model_name = "GradientBoosting_trained_now"

# Predicted probabilities on test set
X_test_s = scaler.transform(X_test)
if hasattr(model, "predict_proba"):
    y_proba = model.predict_proba(X_test_s)[:,1]
else:
    try:
        scores = model.decision_function(X_test_s)
        y_proba = (scores - scores.min())/(scores.max()-scores.min()+1e-9)
    except Exception:
        y_proba = model.predict(X_test_s)

# Select top 5 highest predicted prob (high-stakes)
top5_idx = np.argsort(-y_proba)[:5]
selected = X_test.reset_index(drop=True).loc[top5_idx]
selected_probs = y_proba[top5_idx]

# Try SHAP
shap_spec = importlib.util.find_spec("shap")
created_files = []
if shap_spec is not None:
    try:
        import shap
        # choose explainer
        is_tree = ("XGBoost" in model_name) or ("GradientBoosting" in model.__class__.__name__)
        if is_tree:
            explainer = shap.TreeExplainer(model)
            shap_vals = explainer.shap_values(scaler.transform(selected))
            if isinstance(shap_vals, list):
                shap_vals_to_use = shap_vals[1]
            else:
                shap_vals_to_use = shap_vals
        else:
            background = X_train.sample(n=min(50, len(X_train)), random_state=42)
            explainer = shap.KernelExplainer(lambda x: model.predict_proba(scaler.transform(x))[:,1], background)
            shap_vals_to_use = explainer.shap_values(selected, nsamples=200)
            if isinstance(shap_vals_to_use, list):
                shap_vals_to_use = shap_vals_to_use[1]

        # Save per-case CSV, bar plot PNG, and force plot (HTML)
        for i, idx in enumerate(top5_idx):
            inst = selected.iloc[[i]]
            sv = np.array(shap_vals_to_use[i])
            df_local = pd.DataFrame({
                "feature": X.columns,
                "feature_value": inst.values.flatten(),
                "shap_value": sv
            }).sort_values(by="shap_value", key=lambda s: s.abs(), ascending=False).reset_index(drop=True)
            csv_path = os.path.join(OUT_DIR, f"local_shap_case_{i+1}_idx_{int(idx)}.csv")
            df_local.to_csv(csv_path, index=False); created_files.append(csv_path)
            # bar plot
            plt.figure(figsize=(8,4))
            df_local.set_index("feature")["shap_value"].plot.barh()
            plt.gca().invert_yaxis()
            plt.title(f"Local SHAP values — case {i+1} (test idx {int(idx)})  Pred prob: {selected_probs[i]:.4f}")
            plt.tight_layout()
            png_path = os.path.join(OUT_DIR, f"local_shap_case_{i+1}_idx_{int(idx)}.png")
            plt.savefig(png_path); plt.close()
            created_files.append(png_path)
            # force plot -> save HTML
            try:
                force_html = os.path.join(OUT_DIR, f"local_shap_force_case_{i+1}_idx_{int(idx)}.html")
                exp_value = explainer.expected_value if not isinstance(explainer.expected_value, list) else explainer.expected_value[1]
                fp = shap.force_plot(exp_value, sv, inst, matplotlib=False, show=False)
                shap.save_html(force_html, fp)
                created_files.append(force_html)
            except Exception as e:
                pass
        method = "shap"
    except Exception as e:
        method = "fallback"
else:
    method = "fallback"

# Fallback: approximate local contributions (replace feature with train mean)
if method == "fallback":
    note_path = os.path.join(OUT_DIR, "NOTE_shap_not_available.txt")
    with open(note_path, "w") as f:
        f.write("SHAP not available; used approximate local contributions by replacing each feature with train mean and measuring change in predicted probability.\n")
    created_files.append(note_path)
    for i, idx in enumerate(top5_idx):
        inst = X_test.reset_index(drop=True).loc[idx:idx].copy()
        base_prob = float(model.predict_proba(scaler.transform(inst))[0,1]) if hasattr(model,"predict_proba") else float(model.predict(scaler.transform(inst))[0])
        contributions = []
        for feat in X.columns:
            x_mod = inst.copy()
            x_mod[feat] = X_train[feat].mean()
            mod_prob = float(model.predict_proba(scaler.transform(x_mod))[0,1]) if hasattr(model,"predict_proba") else float(model.predict(scaler.transform(x_mod))[0])
            contributions.append((feat, base_prob - mod_prob, inst[feat].values[0]))
        df_local = pd.DataFrame(contributions, columns=["feature","approx_contribution","feature_value"]).sort_values(by="approx_contribution", key=lambda s: s.abs(), ascending=False)
        csv_path = os.path.join(OUT_DIR, f"local_approx_case_{i+1}_idx_{int(idx)}.csv")
        df_local.to_csv(csv_path, index=False); created_files.append(csv_path)
        plt.figure(figsize=(8,4))
        df_local.set_index("feature")["approx_contribution"].plot.barh()
        plt.gca().invert_yaxis()
        plt.title(f"Local approx contributions — case {i+1} (test idx {int(idx)}) Pred prob: {base_prob:.4f}")
        plt.tight_layout()
        png_path = os.path.join(OUT_DIR, f"local_approx_case_{i+1}_idx_{int(idx)}.png")
        plt.savefig(png_path); plt.close()
        created_files.append(png_path)

# Print brief report
print("Model used:", model_name)
print("Method used:", method)
print("\nSelected cases (test set indices) and predicted probs:")
for i, idx in enumerate(top5_idx):
    print(f" Case {i+1}: test_index={int(idx)}, predicted_prob={selected_probs[i]:.4f}")
print("\nFiles created in", OUT_DIR, ":")
for fn in created_files:
    print(" -", fn)
